# Experiment 3: Implementation of Logistic Regression




### Mathematical Model
$$P(y=1 | x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}$$
- **Binary Cross-Entropy (Log Loss):** $L = -\frac{1}{N} \sum [y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)]$

### Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, auc
)

### Step 2: Plot Sigmoid Curve Intuition & Load Dataset

In [ ]:
z = np.linspace(-10, 10, 200)
sigmoid = 1 / (1 + np.exp(-z))
plt.figure(figsize=(7, 3.5))
plt.plot(z, sigmoid, color='#4F46E5', lw=2.5, label=r'$\sigma(z) = 1 / (1 + e^{-z})$')
plt.axhline(0.5, color='#E11D48', linestyle='--', label='Decision Threshold = 0.5')
plt.axvline(0, color='gray', linestyle=':')
plt.title('Logistic Sigmoid Activation Function', fontweight='bold')
plt.xlabel('z = w^T x + b'); plt.ylabel('Probability P(y=1|x)')
plt.legend(); plt.grid(True, alpha=0.4); plt.show()

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # 0 = Malignant, 1 = Benign
print("Dataset Shape:", X.shape)
print("Class Distribution:", dict(zip(data.target_names, np.bincount(y))))

### Step 3: Preprocessing, Scaling & Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_sc, y_train)

y_pred = clf.predict(X_test_sc)
y_prob = clf.predict_proba(X_test_sc)[:, 1]

### Step 4: Performance Evaluation & Visualizations

In [ ]:
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=data.target_names))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(ax=ax1, cmap='Blues', colorbar=False)
ax1.set_title('Confusion Matrix', fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='#0D9488', lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax2.plot([0, 1], [0, 1], color='gray', linestyle='--')
ax2.set_title('ROC Curve', fontweight='bold')
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.legend(); ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

### Conclusion
- Feature scaling via `StandardScaler` ensures fast convergence of gradient-based optimization in `LogisticRegression`.
- High ROC-AUC and balanced Precision/Recall confirm strong linear separability of benign vs. malignant tumors.